In [2]:
import json

import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd

Embeddings and dataset loading

In [3]:
embeddings = np.load("cbow_embeddings.npy")

X = pd.read_csv('movies_clean.csv', index_col=None)
y = X['label']
X = X.drop('label', axis=1)

Generating mean embeddings for each review in the dataset

In [4]:
with open('word2idx.json', 'r') as file:
    word2idx = json.load(file)

def sentence_embedding(text):
    words = text.split()
    vectors = []
    
    for word in words:
        if word in word2idx:
            vectors.append(embeddings[word2idx[word]])
    
    if len(vectors) == 0:
        return np.zeros(embeddings.shape[1])
    
    return np.mean(vectors, axis=0)

X['embedding'] = X['text'].apply(sentence_embedding)
X = X.drop('text', axis=1)

Preparing pytorch dataset

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(X['embedding'],
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp,
                                                y_temp,
                                                test_size=0.5,
                                                random_state=42)

X_train = np.stack(X_train.values)
X_test = np.stack(X_test.values)
X_val = np.stack(X_val.values)
y_train = np.stack(y_train.values)
y_test = np.stack(y_test.values)
y_val = np.stack(y_val.values)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.float32))

test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                             torch.tensor(y_test, dtype=torch.float32))

validation_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                                   torch.tensor(y_val, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False)

INPUT_SIZE = embeddings.shape[1]

Training MLP model

In [ ]:
from sequential_nn import SentimentNN


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentimentNN(INPUT_SIZE, [128, 128])
model = model.to(device)
model.train()

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(12):
    total_loss = 0

    for example, label in train_loader:
        example, label = example.to(device), label.to(device)
        optimizer.zero_grad()
        output = model(example).squeeze(1)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, loss: {total_loss:.4f}")
    model.eval()

    with torch.no_grad():
        correct = 0
        total = 0
        for example, label in test_loader:
            example, label = example.to(device), label.to(device)
            output = model(example).squeeze(1)
            preds = torch.sigmoid(output) >= 0.5
            correct += (preds == label).sum().item()
            total += label.size(0)

        print("Test Accuracy:", correct / total)

    model.train()

Epoch 1, loss: 192.6709
Test Accuracy: 0.8468851638729047
Epoch 2, loss: 172.0505
Test Accuracy: 0.8458844133099825
Epoch 3, loss: 167.6755
Test Accuracy: 0.8498874155616712
Epoch 4, loss: 163.7867
Test Accuracy: 0.8506379784838629
Epoch 5, loss: 160.1182
Test Accuracy: 0.8526394796097073
Epoch 6, loss: 156.0799
Test Accuracy: 0.8506379784838629
Epoch 7, loss: 152.7688
Test Accuracy: 0.8516387290467851
Epoch 8, loss: 148.6812
Test Accuracy: 0.8503877908431323
Epoch 9, loss: 144.5328
Test Accuracy: 0.8503877908431323
Epoch 10, loss: 140.7708
Test Accuracy: 0.8476357267950964
Epoch 11, loss: 136.7981
Test Accuracy: 0.8521391043282461
Epoch 12, loss: 132.4516
Test Accuracy: 0.8436327245434075


Comment:
16 epochs leads to overfitting. Should try less

Evaluate on validation set

In [ ]:
model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    
    for example, label in validation_loader:
        example, label = example.to(device), label.to(device)
        output = model(example).squeeze(1)
        preds = torch.sigmoid(output) >= 0.5
        correct += (preds == label).sum().item()
        total += label.size(0)

    print("Test Accuracy:", correct / total)


Test Accuracy: 0.8421315986990243


# RNN model

Loading dataset with text sequence embeddings

In [13]:
embeddings = np.load("cbow_embeddings.npy")

X = pd.read_csv('movies_clean.csv', index_col=None)
y = X['label']
X = X.drop('label', axis=1)

with open('word2idx.json', 'r') as file:
    word2idx = json.load(file)

def sequence_embedding(text):
    words = text.split()
    vectors = []
    for word in words:
        if word in word2idx:
            vectors.append(embeddings[word2idx[word]])
    return vectors

X['embedding'] = X['text'].apply(sequence_embedding)
X = X.drop('text', axis=1)

lengths_all = X['embedding'].apply(len)
MAX_LEN = int(np.quantile(lengths_all, 0.95))

def pad_sequence(seq, max_len, embed_dim):
    padded = np.zeros((max_len, embed_dim), dtype=np.float32)
    length = min(len(seq), max_len)
    if length > 0:
        padded[:length] = seq[:length]
    return padded, length

embed_dim = embeddings.shape[1]

def prepare(sequences):
    padded, lengths = zip(*[pad_sequence(s, MAX_LEN, embed_dim) for s in sequences])
    return np.stack(padded), np.array(lengths, dtype=np.int64)

X_train, X_temp, y_train, y_temp = train_test_split(X['embedding'], y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

X_train, lengths_train = prepare(X_train.values)
X_val, lengths_val = prepare(X_val.values)
X_test, lengths_test = prepare(X_test.values)

y_train = y_train.values.astype(np.float32)
y_val = y_val.values.astype(np.float32)
y_test = y_test.values.astype(np.float32)

def make_loader(X, lengths, y, batch_size, shuffle, num_workers, pin_memory):
    dataset = TensorDataset(
        torch.tensor(X),
        torch.tensor(lengths),
        torch.tensor(y)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=pin_memory)

train_loader = make_loader(X_train, lengths_train, y_train, 64, shuffle=True, num_workers=4, pin_memory=True)
test_loader = make_loader(X_test,  lengths_test,  y_test,  64, shuffle=False, num_workers=4, pin_memory=True)
validation_loader = make_loader(X_val,   lengths_val,   y_val,   64, shuffle=False, num_workers=4, pin_memory=True)

INPUT_SIZE = embed_dim

Training RNN model

In [ ]:
from recurrent_nn import RecurrentNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RecurrentNN(INPUT_SIZE, hidden_size=64)
model.to(device)

criterion = nn.BCELoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

for epoch in range(15):
    model.train()
    total_loss = 0

    for example, lengths, label in train_loader:
        example, label = example.to(device), label.to(device)
        optimizer.zero_grad()
        output = model(example, lengths).squeeze(1)
        loss = criterion(output, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, loss: {total_loss:.4f}")
    model.eval()

    with torch.no_grad():
        correct = 0
        total = 0
        for example, lengths, label in test_loader:
            example, label = example.to(device), label.to(device)
            output = model(example, lengths).squeeze(1)
            preds = output >= 0.5
            correct += (preds == label).sum().item()
            total += label.size(0)

        print("Test Accuracy:", correct / total)


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1, loss: 246.6305
Test Accuracy: 0.7773329997498124


Evaluating on validation set

In [16]:
model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    
    for example, lengths,  label in validation_loader:
        example, label = example.to(device), label.to(device)
        output = model(example, lengths).squeeze(1)
        preds = output >= 0.5
        correct += (preds == label).sum().item()
        total += label.size(0)

    print("Test Accuracy:", correct / total)


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Accuracy: 0.7875906930197648
